In [1]:
!pip install networkx -q
!pip install scipy -q
# !pip install ipycytoscape -q

In [2]:
from utils.distances_database_manager import get_distances, get_and_pivot_distances, get_data_with_max_distance
import dask
import dask.dataframe as dd
import pandas as pd
from IPython.display import display, Markdown
import networkx as nx
from tqdm import tqdm
import scipy as sp
from statistics import mean
from dask.diagnostics import ProgressBar
from dask.distributed import Client, progress, LocalCluster
from dask import delayed

In [11]:
# paths
database_path = '../data_parquet_starpep/'
base_path = '../output/StarPep/PaperNew/2024-12-23T15.40.45-Data_Ingestion/Analysis/'
output_path = './output/graphics/'

In [4]:
def build_graph_from_group(group, distance_function):
    # Crear el grafo
    nx_graph = nx.Graph()

    # Establecer atributos generales del grafo
    sequence = group['sequence'].iloc[0]
    nx_graph.graph['sequence'] = sequence

    # Añadir nodos y aristas
    for _, row in group.iterrows():
        source = row['aminoacid_source']
        target = row['aminoacid_target']
        weight = row[distance_function]
        nx_graph.add_edge(source, target, weight=weight)

    return nx_graph

def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum().compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data

In [5]:
distance_function = 'euclidean'

lim_min = 0 
lim_max = 10  
filters = [(distance_function, '>=', lim_min), (distance_function, '<=', lim_max)]

data = dd.read_parquet(
    database_path,
    columns=['sequence', 'aminoacid_source', 'aminoacid_target', distance_function],
    filters=filters
)

data = repartition_data(data, 100)

# result.info(memory_usage="deep")

In [ ]:
from dask.diagnostics import ProgressBar

# Activar la barra de progreso
with ProgressBar():
    graphs = data.groupby('sequence').apply(
        # build_graph_from_group, meta=('graph', 'object')
        lambda group: add_edges_to_graph(group, distance_function), 
        meta=('graph', 'object')
    ).compute()

# Convertir el resultado a una lista de grafos
graphs_list = list(graphs)

In [ ]:
from dask import delayed
import dask.dataframe as dd

distance_function = 'euclidean'

# Crear una función retardada
@delayed
def add_edges_to_graph(group, distance_function):
    graph = nx.Graph()
    graph.graph['sequence'] = group['sequence'].iloc[0]

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values
    weights = group[distance_function].values
    
    graph.add_weighted_edges_from(zip(sources, targets, weights))
    
    return graph


@delayed
def compute_graph_metrics(group, distance_function):
    graph = nx.Graph(to_undirected=True)

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    metrics = {
        'sequence': group['sequence'].iloc[0],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph)),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph)),
        'closeness_centrality': mean(nx.closeness_centrality(graph)),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph)),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph)) 
    }   

    return metrics

In [ ]:
%%time
with ProgressBar():
    df = data.compute() 
    tasks = [compute_graph_metrics(group, 'euclidean') for _, group in df.groupby(['sequence'])]

In [ ]:
%%time
with ProgressBar():
     results = dask.compute(*tasks)

In [ ]:
a = pd.DataFrame(results)

In [ ]:
%%time
with ProgressBar():
    df = data.compute() 
    tasks = [compute_graph_metrics(group, 'euclidean') for _, group in df.groupby(['sequence'])]

In [ ]:
dask.visualize(*tasks, engine='ipycytoscape', filename='graph.pdf')

In [12]:
def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum().compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data
    
def load_data(distance_function, interval):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[0])]
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    data = repartition_data(data, 100)

    grouped = data.groupby('sequence')
    
    return grouped

def construct_graph(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    return graph
    

In [13]:
distance_function = 'euclidean'
interval = (0, 10)
data = load_data(distance_function, interval)

In [15]:
data

In [ ]:
def construct_graph(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    return graph

def union_edges(G1, G2):
    nx.union_all([G1, G2])

def compute_graph_metrics(graph):
    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()),
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }   

    return metrics

def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum().compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data

def load_data(distance_function, interval):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[0])]
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    data = repartition_data(data, 100)

def compute_metrics(data, filter):
    distance_function = filter['distance_function']
    interval = filter['interval']

    # load_data
    data = load_data(distance_function, interval)

    # filter data
    data_partition = data[(data[distance_function] >= interval[0]) & (data[distance_function] <= interval[1])]
    
    # group data
    metrics = dd.Aggregation(
        name='compute_graph_metrics',
        chunk=lambda g: (g.construct_graph()),
        agg=lambda construct_graph: (construct_graph.union_edges()),
        finalize=lambda construct_graph: compute_graph_metrics(union_edges),
    )  
    
    data_partition.groupby('sequence').agg(metrics)  


filters = [
    {'distance_function': 'euclidean', "interval": (0, 20)},
    {'distance_function': 'euclidean', "interval": (0, 15)},
    {'distance_function': 'euclidean', "interval": (0, 10)} 
]

tasks = []
for filter in filters:
    task = compute_metrics(data, filter)
    tasks.append(task)

results = tasks.compute()

In [ ]:
import networkx as nx
import dask.dataframe as dd
from statistics import mean

def construct_graph(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    return graph

def union_edges(G1, G2):
    # Return the union of two graphs
    return nx.union(G1, G2)

def compute_graph_metrics(graph):
    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()),
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }
    return metrics

def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum().compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data

def load_data(distance_function, interval):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    
    data = repartition_data(data, 100)
    return data

# La función que se utiliza en la agregación
def graph_aggregation(group, distance_function, interval):
    # Construye el gráfico para el grupo dado
    return construct_graph(group, distance_function, interval)

# Agregación en Dask
def compute_metrics(data, filter):
    distance_function = filter['distance_function']
    interval = filter['interval']

    # Cargar datos
    data = load_data(distance_function, interval)

    # Crear la agregación de Dask
    graph_aggregation_dask = dd.Aggregation(
        name='compute_graph',
        chunk=lambda group: graph_aggregation(group, distance_function, interval),  # Construcción de los gráficos por partición
        agg=union_edges,  # Unión de los gráficos en la fase de agregación
        finalize=lambda graph: compute_graph_metrics(graph)  # Cálculo de las métricas del gráfico final
    )

    # Realizar la agregación
    results = data.groupby('sequence').agg(graph_aggregation_dask)

    return results

# Filtros de ejemplo
filters = [
    {'distance_function': 'euclidean', "interval": (0, 20)}#,
    #{'distance_function': 'euclidean', "interval": (0, 15)},
    #{'distance_function': 'euclidean', "interval": (0, 10)} 
]

# Procesamiento de filtros
tasks = []
for filter in filters:
    task = compute_metrics(data, filter)
    tasks.append(task)

# Ejecutar todo
#with ProgressBar():
#    results = dd.compute(*tasks)

In [ ]:
import dask.dataframe as dd
import pandas as pd
import networkx as nx

# Función para construir un grafo a partir de un grupo
def construct_graph(group):
    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values
    edges = list(zip(sources, targets))
    return {'nodes': list(set(sources) | set(targets)), 'edges': edges}

# Función para combinar los grafos de diferentes particiones
def union_edges(g1, g2):
    nodes = set(g1['nodes']) | set(g2['nodes'])
    edges = set(g1['edges']) | set(g2['edges'])
    return {'nodes': list(nodes), 'edges': list(edges)}

# Función para calcular métricas finales a partir del grafo combinado
def finalize(graph):
    g = nx.Graph()
    g.add_edges_from(graph['edges'])
    return compute_graph_metrics(g)

# Función para calcular métricas del grafo
def compute_graph_metrics(graph):
    return {
        'num_nodes': graph.number_of_nodes(),
        'num_edges': graph.number_of_edges(),
        'density': nx.density(graph)
    }

# Definir la agregación personalizada para Dask
graph_aggregation_dask = dd.Aggregation(
    name='compute_graph',
    chunk=lambda df: pd.DataFrame([construct_graph(df)], index=[0]),  # Agregar índice explícito
    agg=lambda dfs: pd.DataFrame([union_edges(*dfs)], index=[0]),     # Agregar índice explícito
    finalize=lambda df: pd.DataFrame([finalize(df.iloc[0])], index=[0]),  # Agregar índice explícito
)

# Función para repartir datos en particiones óptimas
def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum().compute()
    desired_partition_size = partition_size * 1e6
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data

# Función para cargar datos desde archivos Parquet
def load_data(database_path, distance_function, interval):
    filters = [(distance_function, '>=', interval[0]), (distance_function, '<=', interval[1])]
    data = dd.read_parquet(
        database_path,
        columns=['sequence', 'aminoacid_source', 'aminoacid_target'],
        filters=filters
    )
    data = repartition_data(data, 100)
    return data

# Cargar datos (ajusta la ruta a tu base de datos Parquet)
data = load_data(database_path, 'euclidean', (0, 10))

# Realizar la agregación
results = data.groupby('sequence').agg(graph_aggregation_dask)

# Ejecutar el cálculo y obtener los resultados finales
with Processbar():
    final_results = results.compute()
print(final_results)



In [39]:
graph_aggregation_dask

# THIS

In [4]:
%%time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

def repartition_data(data, partition_size):
    total_size = data.memory_usage(deep=True).sum()
    if isinstance(total_size, dd.core.Scalar):  # Verificar si es un objeto Dask
        total_size = total_size.compute()
    desired_partition_size = partition_size * 1e6  # 200 MB
    optimal_partitions = max(1, int(total_size / desired_partition_size))
    data = data.repartition(npartitions=optimal_partitions)
    data = data.persist()
    return data
    
def construct_graph(group, distance_function, interval):
    graph = nx.Graph(to_undirected=True)

    graph.graph['sequence'] = group['sequence'].iloc[0]
    graph.graph['distance_function'] = distance_function
    graph.graph['interval'] = interval

    sources = group['aminoacid_source'].values
    targets = group['aminoacid_target'].values    
    graph.add_edges_from(zip(sources, targets))

    return graph

def compute_graph_metrics(graph):
    metrics = {
        'sequence': graph.graph['sequence'],
        'distance_function': graph.graph['distance_function'],
        'interval': graph.graph['interval'],
        'number_of_nodes': graph.number_of_nodes(),
        'number_of_edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'degree_centrality': mean(nx.degree_centrality(graph).values()),
        'eigenvector_centrality': mean(nx.eigenvector_centrality_numpy(graph).values()),
        'closeness_centrality': mean(nx.closeness_centrality(graph).values()),
        'betweenness_centrality': mean(nx.betweenness_centrality(graph).values()),
        'harmonic_centrality': mean(nx.harmonic_centrality(graph).values())
    }   

    return metrics


def create_group(data, distance_function, interval):
    # filter data
    data_partition = data[(data[distance_function] >= interval[0]) & (data[distance_function] <= interval[1])]
    
    # group data
    grouped = data_partition.groupby('sequence')

    return grouped

@delayed
def process_filter(data, filter):
    distance_function = filter['distance_function']
    interval = filter['interval']

    # filter data
    data_partition = data[(data[distance_function] >= interval[0]) & (data[distance_function] <= interval[1])]
    
    # group data
    grouped = data_partition.groupby('sequence')
    
    results = []
    
    # process group
    for sequence, group in grouped:
        # construct graphs
        graph = construct_graph(group, distance_function, interval)
        
        # compute metrics
        metrics = compute_graph_metrics(graph)
        results.append(metrics)
    
    return results

#config client
#client = Client(processes=True)  # Configurar Dask para usar múltiples procesos
#cluster = LocalCluster()
#client = Client(cluster)

# data
data = dd.read_parquet(database_path)

# partitions
filters = [
    {'distance_function': 'euclidean', "interval": (0, 20)},
    {'distance_function': 'euclidean', "interval": (0, 15)},
    {'distance_function': 'euclidean', "interval": (0, 10)} 
]

# apply filter to data 
#config client
#client = Client(processes=True)

#num_cores = multiprocessing.cpu_count()
#with dask.config.set(pool=ProcessPoolExecutor(num_cores)):
tasks = []
for filter in filters:
    task = process_filter(data, filter)
    tasks.append(task)

# Execute in parallel 
dask.config.set(scheduler='processes') 

with ProgressBar():
    computed_metrics = dask.compute(*tasks)

#client.close()

[########################################] | 100% Completed | 33m 25s
CPU times: user 47.6 s, sys: 47.9 s, total: 1min 35s
Wall time: 33min 25s


In [ ]:
computed_metrics

In [15]:
from dask.distributed import LocalCluster
from tqdm import tqdm

client = Client()

@delayed
def process_filter(data, filter):
    distance_function = filter['distance_function']
    interval = filter['interval']

    # group data
    grouped = create_group(data, distance_function, interval)

    # process group
    results = []   
    for sequence, group in grouped:
        # construct graphs
        graph = construct_graph(group, distance_function, interval)
        
        # compute metrics
        metrics = compute_graph_metrics(graph)
        results.append(metrics)
    
    return results
    
# Submit work to happen in parallel
tasks = []
for filter in filters:
    task = process_filter(data, filter)
    tasks.append(task)

# Gather results back to local computer
with ProgressBar():
    results = client.compute(tasks) 

client.close()

/opt/conda/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43903 instead
  warnings.warn(
2025-01-05 04:23:45,275 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('read_parquet-e5b8d659e0dd8dfcce0ea11ef3571f86', 3))" coro=<Worker.execute() done, defined at /opt/conda/lib/python3.11/site-packages/distributed/worker_state_machine.py:3606>> ended with CancelledError
2025-01-05 04:23:45,294 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('read_parquet-e5b8d659e0dd8dfcce0ea11ef3571f86', 1))" coro=<Worker.execute() done, defined at /opt/conda/lib/python3.11/site-packages/distributed/worker_state_machine.py:3606>> ended with CancelledError
2025-01-05 04:23:45,306 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('read_pa

In [ ]:
# Aplicar la función retardada a los grupos
with ProgressBar():
    metrics = data.groupby('sequence').apply(
        lambda group: compute_graph_metrics(group, distance_function), 
        meta=('metrics', 'object')
    ).compute()

In [ ]:
# Aplicar la función retardada a los grupos
with ProgressBar():
    graphs = data.groupby('sequence').apply(
        lambda group: add_edges_to_graph(group, distance_function), 
        meta=('graph', 'object')
    ).compute()

In [ ]:
# Aplicar la función retardada a los grupos
with ProgressBar():
    metrics = data.groupby('sequence').apply(
        lambda group: compute_graph_metrics(group, distance_function), 
        meta=('metrics', 'object')
    ).compute()

In [ ]:
%%time
with ProgressBar():
    result = dask.compute(metrics)

In [ ]:
a = graphs[0].compute()
a.number_of_edges()
a.graph

In [ ]:
graphs_list = graphs[0].compute()
first_graph = graphs_list[0]
graphs_list.graph

In [ ]:
grouped = result.groupby('sequence')

graphs = []
for sequence, group in tqdm(grouped, desc="Processing sequences"):
    nx_graph = nx.Graph()
    nx_graph.graph['sequence'] = sequence
    nx_graph.graph['distance_functions'] = distance_function
    
    for _, row in group.iterrows():
        source = row['aminoacid_source']
        target = row['aminoacid_target']
        distance = row[distance_function]
        
        nx_graph.add_edge(source, target, weight=distance)
    
    graphs.append(nx_graph)

In [ ]:
graphs[0].number_of_nodes()

In [ ]:
graphs[0].number_of_edges()

In [ ]:
nx.density(graphs[0])

In [ ]:
# Leer el archivo Parquet
data = dd.read_parquet(
    database_path,
    columns=['sequence', 'aminoacid_source', 'aminoacid_target', distance_function],
    filters=filters
)

# Agrupar los datos por la columna 'sequence'
grouped = data.groupby('sequence')

# Procesar cada grupo usando `.to_delayed()` y tqdm para mostrar una barra de progreso
graphs = []
for delayed_chunk in tqdm(grouped.to_delayed(), desc="Procesando grupos"):
    chunk = delayed_chunk.compute()  # Computar el fragmento de datos actual

    # Iterar sobre cada secuencia en el grupo
    for sequence, group_data in chunk.groupby('sequence'):
        # Crear un grafo para cada secuencia
        nx_graph = nx.Graph()
        nx_graph.graph['sequence'] = sequence
        nx_graph.graph['distance_functions'] = distance_function

        # Agregar nodos y aristas al grafo
        for _, row in group_data.iterrows():
            source = row['aminoacid_source']
            target = row['aminoacid_target']
            distance = row[distance_function]
            nx_graph.add_edge(source, target, weight=distance)

        # Guardar el grafo en la lista
        graphs.append(nx_graph)

In [17]:
import dask.dataframe as dd
import pandas as pd
import numpy as np

# Crear un DataFrame pequeño en Pandas
data = {
    'g': ['A', 'A', 'B', 'B', 'C', 'C', 'A', 'B', 'C'],
    'value': [1, 2, 3, 4, 5, 6, 7, 8, 9]
}

# Convertirlo en un DataFrame de Dask
df = dd.from_pandas(pd.DataFrame(data), npartitions=2)

# Definir la agregación personalizada (en este caso una suma)
custom_sum = dd.Aggregation(
    name='custom_sum',
    chunk=lambda s: s.sum(),  # Operación de agregación en el chunk
    agg=lambda s0: s0.mean()    # Agregación final de todos los chunks
)

# Aplicar la agregación al DataFrame agrupado por la columna 'g'
result = df.groupby('g')['value'].agg(custom_sum)

# Computar los resultados
print("Resultados de la agregación personalizada:")
print(result.compute())  # Para obtener el resultado final como un DataFrame Pandas



Resultados de la agregación personalizada:
g
A     5.0
B     7.5
C    10.0
Name: value, dtype: float64


In [21]:
import pandas as pd
import networkx as nx
import numpy as np
import dask.dataframe as dd
from dask.dataframe import Aggregation

# Datos de ejemplo
data = {
    'sequence': ['ABC', 'ABC', 'DEF', 'DEF', 'ABC'],
    'peptide': ['AGCT', 'GCTA', 'AGCT', 'CTAG', 'TACG']
}

# Crear DataFrame
df = pd.DataFrame(data)

# Función para construir el grafo y calcular métricas
def construct_graph_and_compute_metrics(group):
    # Construcción del grafo
    G = nx.Graph()
    for seq in group['peptide']:
        for i in range(len(seq) - 1):
            G.add_edge(seq[i], seq[i + 1])

    # Cálculo de métricas
    metrics = {
        'sequence': group['sequence'].iloc[0],  # Usamos la primera secuencia como ejemplo
        'number_of_nodes': len(G.nodes),
        'number_of_edges': len(G.edges),
        'density': nx.density(G),
        'degree_centrality': np.mean(list(nx.degree_centrality(G).values())),
        'eigenvector_centrality': np.mean(list(nx.eigenvector_centrality(G).values())),
        'closeness_centrality': np.mean(list(nx.closeness_centrality(G).values())),
        'betweenness_centrality': np.mean(list(nx.betweenness_centrality(G).values())),
        'harmonic_centrality': np.mean(list(nx.harmonic_centrality(G).values())),
    }
    return pd.Series(metrics)

# Convertir a Dask DataFrame
ddf = dd.from_pandas(df, npartitions=2)

# Agrupar por 'sequence' y aplicar la función de métricas
result = ddf.groupby('sequence').apply(construct_graph_and_compute_metrics, meta={
    'sequence': 'object', 
    'number_of_nodes': 'int64',
    'number_of_edges': 'int64',
    'density': 'float64',
    'degree_centrality': 'float64',
    'eigenvector_centrality': 'float64',
    'closeness_centrality': 'float64',
    'betweenness_centrality': 'float64',
    'harmonic_centrality': 'float64',
})

# Computar el resultado
with ProgressBar():
    final_result = result.compute()
print(final_result)


[########################################] | 100% Completed | 1.63 ss
         sequence  number_of_nodes  number_of_edges   density  \
sequence                                                        
ABC           ABC                4                5  0.833333   
DEF           DEF                4                4  0.666667   

          degree_centrality  eigenvector_centrality  closeness_centrality  \
sequence                                                                    
ABC                0.833333                0.496254                 0.875   
DEF                0.666667                0.500000                 0.750   

          betweenness_centrality  harmonic_centrality  
sequence                                               
ABC                     0.083333                 2.75  
DEF                     0.166667                 2.50  
